# 5.13 — Understanding Supervised Learning Problem Types

This repo uses supervised learning: we learn from **features** $X$ and known **targets** $y$.

Before choosing models or metrics, you must identify what kind of problem $y$ represents:

- **Classification**: predict a **category** (discrete labels)
  - **Binary** (2 classes)
  - **Multi-class** (3+ mutually exclusive classes)
  - **Multi-label** (multiple labels can apply at once)
- **Regression**: predict a **number** (continuous or count)
  - **Continuous** (real-valued)
  - **Count** (0,1,2,...)

The problem type determines:
- which algorithms make sense
- which metrics are meaningful
- what ‘success’ means

## 1) A small helper: infer the supervised problem type

This repo includes `src/problem_type.py` to infer problem type from a target column and recommend metrics.
It’s a heuristic guardrail: it helps you avoid training the wrong type of model for a given target.

In [ ]:
import pandas as pd

from src.data_loader import load_csv
from src.problem_type import infer_supervised_problem_type, recommended_metrics

df = load_csv('data/raw/source_demo_crops_20260321.csv')
df

## 2) Classification example (binary)

Our training pipeline uses a **binary target** derived from `yield_kg` (median split):
- 1 = yield is high
- 0 = yield is low

That turns a continuous-ish quantity into a **classification** task (useful for demonstration, but note it discards magnitude).

In [ ]:
y_binary = (df['yield_kg'] >= df['yield_kg'].median()).astype(int)
ptype = infer_supervised_problem_type(y_binary)
ptype, recommended_metrics(ptype)

## 3) Regression example (continuous)

If you predict `yield_kg` directly, that is **regression**: the output is a number and errors have magnitude.
Typical metrics: MAE, RMSE, $R^2$.

In [ ]:
y_reg = df['yield_kg']
ptype = infer_supervised_problem_type(y_reg)
ptype, recommended_metrics(ptype)

## 4) Multi-class example (toy)

If the target is something like `crop` (Rice/Wheat/Maize/Cotton), that is **multi-class classification**.

In [ ]:
y_multiclass = df['crop']
ptype = infer_supervised_problem_type(y_multiclass)
ptype, recommended_metrics(ptype)

## 5) Why this matters in the pipeline

The training pipeline now validates that the target looks like **classification** before training a classifier.
This prevents a common failure mode: accidentally treating a regression target as classification (or vice versa).

Try passing `--target-column yield_kg` to training to see the guardrail trigger (because `yield_kg` is regression-like).

In [ ]:
# This cell is optional; it demonstrates the guardrail behavior.
# It will raise a ValueError because yield_kg is inferred as regression-like.
from src.training_pipeline import run_training_pipeline

try:
    run_training_pipeline(data_path='data/raw/source_demo_crops_20260321.csv', target_column='yield_kg')
except ValueError as exc:
    print('Guardrail triggered:', exc)